In [43]:
from pathlib import Path
import pandas as pd

DATA = Path.home() / "Desktop/mimic-iv-clinical-database-demo-2/hosp"

labs = pd.read_json("data/labs_all.json")
tx = pd.read_json("data/tx_all.json")
labs["charttime"] = pd.to_datetime(labs["charttime"])   
tx["charttime"] = pd.to_datetime(tx["charttime"])

adm = pd.read_csv(DATA / "admissions.csv",
                  usecols=["hadm_id", "admittime", "dischtime"],
                  parse_dates=["admittime", "dischtime"])
print(len(labs), len(tx), len(adm))

BASELINE_WINDOW_HOURS = 24      # baseline：fist reading after admission wihthin 24 hours
TREATMENT_WINDOW_HOURS = 2      # onset: after onset, treatment within 2 hours 

2719 386 275


In [44]:
# onlu keep labs and tx that are within the hospital stay window
k_with_adm = labs.merge(adm, on="hadm_id", how="inner")
in_stay = (k_with_adm["charttime"] >= k_with_adm["admittime"]) & (k_with_adm["charttime"] <= k_with_adm["dischtime"])
k_in_stay = k_with_adm[in_stay].copy()
k_in_stay["hours_from_admit"] = (k_in_stay["charttime"] - k_in_stay["admittime"]).dt.total_seconds() / 3600
print(len(k_in_stay), k_in_stay["hadm_id"].nunique())

2664 243


In [45]:
# filter for 24 hours and normal range
baseline = (
    k_in_stay[k_in_stay["hours_from_admit"] <= 24]
    .sort_values("charttime")
    .drop_duplicates("hadm_id", keep="first")         
)          # first reading in 24 hours
normal_baseline = baseline[(baseline["valuenum"] >= 3.5) & (baseline["valuenum"] <= 5.0)]   # 3.5–5.0 
print(len(baseline), len(normal_baseline))

231 183


In [46]:
# filter for hyperkalemia events
nb = normal_baseline[["hadm_id", "charttime"]].rename(columns={"charttime": "baseline_time"})
candidates = k_in_stay.merge(nb, on="hadm_id", how="inner")
hyperk_events = candidates[ (candidates["valuenum"] > 5.0) & (candidates["charttime"] > candidates["baseline_time"]) ]
hyperk_events = hyperk_events.rename(columns={"charttime": "onset_time", "valuenum": "onset_k"})
print(len(hyperk_events), hyperk_events["hadm_id"].nunique())

116 39


In [47]:
# only keep calicum and insulin treatment
core = (
    tx[tx["drug_class"].isin(["Calcium", "Insulin", "Furosemide", "Bicarbonate"])]
    .rename(columns={"charttime": "med_time"})
    [["hadm_id", "med_time", "drug_class"]]
    .sort_values("med_time")
)
len(core)

351

In [48]:
# merge hyperkalemia events with treatment 
left = hyperk_events.sort_values("onset_time")      

matched = pd.merge_asof(
    left, core,
    left_on="onset_time",
    right_on="med_time",
    by="hadm_id",
    direction="forward",
    tolerance=pd.Timedelta(hours=2),
)
treated = matched[matched["med_time"].notna()]      # only keep those with treatment
print(len(treated), treated["hadm_id"].nunique())

8 3


In [49]:
# only get the first onset
cohort = treated.sort_values("onset_time").drop_duplicates("hadm_id")
cohort[["hadm_id", "onset_time", "onset_k", "med_time", "drug_class"]]

,hadm_id,onset_time,onset_k,med_time,drug_class
7,29276678,2116-03-01 23:56:00,5.4,2116-03-02 01:23:00,Calcium
70,26486158,2148-08-29 12:05:00,5.2,2148-08-29 12:15:00,Furosemide
98,20297618,2180-02-21 17:15:00,5.5,2180-02-21 17:48:00,Insulin


In [50]:
# highest K for each admission
summary = labs.groupby("hadm_id")["valuenum"].max().reset_index(name="max_k")

# merge with phenotype cohort
summary = summary.merge(
    cohort[["hadm_id", "onset_time", "onset_k", "med_time", "drug_class"]],
    on="hadm_id", how="left",
)

# add new columns for phenotype and treatment window
summary["phenotype"] = summary["onset_time"].notnull()
summary["window_end"] = summary["onset_time"] + pd.Timedelta(hours=TREATMENT_WINDOW_HOURS)

# admissions whose first potassium (within 24 h) was already above 5.0
high_on_adm = baseline[baseline["valuenum"] > 5.0]["hadm_id"]                                   
summary["high_k_on_admission"] = summary["hadm_id"].isin(high_on_adm)
print(summary["high_k_on_admission"].sum())
# export to json in highest K order
summary = summary.sort_values("max_k", ascending=False)
summary.to_json("data/admissions.json", orient="records", date_format="iso")
print(len(summary), summary["phenotype"].sum())

23
250 3


## Heatmap array

In [51]:
counts = tx.pivot_table(
    index="hadm_id",          # one row per admission
    columns="drug_class",     # one column per meds
    values="charttime",       
    aggfunc="count",          
    fill_value=0,             
)
print(counts.shape)
counts.head()

(62, 5)


drug_class,Bicarbonate,Calcium,Dextrose 50%,Furosemide,Insulin
hadm_id,,,,,
20214994,0,0,0,4,0
20297618,0,1,1,7,1
20364112,0,3,1,17,0
20755971,0,0,0,7,0
21133938,1,1,0,4,1


In [52]:
# only keep admissions with hyperkalemia events
high = summary[summary["max_k"] > 5.0]
len(high)

74

In [53]:
# align counts with high admissions and only keep the three drug classes
counts = counts.reindex(
    index=high["hadm_id"],                            # row：74 admissions, in the order of high
    columns=["Calcium", "Insulin", "Dextrose 50%", "Furosemide", "Bicarbonate"],   # column：5 drug classes
    fill_value=0,                                     
)
print(counts.shape)

(74, 5)


In [54]:
import json

heat = {
    "x": list(counts.columns),                      # ["Calcium", "Insulin", "Dextrose 50%"]
    "y": [str(h) for h in counts.index],            # hadm_id into string
    "z": counts.values.tolist(),                    # [[0, 0, 1], [5, 0, 0], ...]
    "max_k": high["max_k"].tolist(),
}
Path("data/heatmap.json").write_text(json.dumps(heat))

2618